# Phase 0-3: RF-DETR Nano 파인튜닝 검증

**목적**: 오토라벨(Grounding DINO 자동 생성 박스)만으로 학습한 소형 모델이 GT 기준 어디까지 가는지 확인.

**사용법**: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 → 아래 셀 순서대로 실행. `rfdetr_person.zip`을 업로드하라는 셀에서 파일 선택.

- train/valid = 오토라벨 (사람 손 0)
- test = coco128 사람 수작업 GT — 진짜 정답 기준 평가

In [ ]:
!nvidia-smi -L
%pip install -q rfdetr supervision

In [ ]:
# 데이터셋 업로드 (rfdetr_person.zip)
from google.colab import files
up = files.upload()
!unzip -oq rfdetr_person.zip -d dataset && ls dataset

In [ ]:
# 파인튜닝 — 제로컨피그 기본값 (T4 기준 약 20~40분)
import time
from rfdetr import RFDETRNano

model = RFDETRNano()
t0 = time.time()
model.train(dataset_dir="dataset", epochs=30, batch_size=4, grad_accum_steps=4, lr=1e-4)
print(f"학습 시간: {(time.time()-t0)/60:.1f}분")

In [ ]:
# GT 테스트셋 평가 (mAP50)
import json, supervision as sv
import numpy as np
from pathlib import Path
from PIL import Image

test_dir = Path("dataset/test")
coco = json.loads((test_dir / "_annotations.coco.json").read_text())
gt_by_img = {}
for a in coco["annotations"]:
    gt_by_img.setdefault(a["image_id"], []).append(a["bbox"])

targets, predictions = [], []
for im in coco["images"]:
    img = Image.open(test_dir / im["file_name"])
    det = model.predict(img, threshold=0.4)
    predictions.append(det)
    boxes = np.array([[x, y, x + w, y + h] for x, y, w, h in gt_by_img.get(im["id"], [])]).reshape(-1, 4)
    targets.append(sv.Detections(xyxy=boxes, class_id=np.zeros(len(boxes), dtype=int)))

map_result = sv.MeanAveragePrecision.from_detections(predictions=predictions, targets=targets)
print(f"파인튜닝 모델 (오토라벨로만 학습) — GT 기준 mAP50: {map_result.map50:.3f}, mAP50-95: {map_result.map50_95:.3f}")
print("비교용 제로샷 Grounding DINO 베이스라인은 로컬에서 측정한 값과 대조")

In [ ]:
# 가중치 다운로드 (로컬 배포용)
from google.colab import files as gf
import glob
ckpts = glob.glob("output/**/checkpoint_best_total.pth", recursive=True) or glob.glob("output/**/*.pth", recursive=True)
print(ckpts)
if ckpts:
    gf.download(ckpts[0])